# Sorani (Kurdish) Voice Training - Piper TTS (Seçenek A)

Bu not defteri, kadın Sorani (ckb) sesini Piper TTS'e eğitir. Veri kaynağı: Common Voice ckb (HuggingFace) üzerinden. Eğitim sonunda model.onnx + model.onnx.json dosyaları üretilir.

Adimlar: 1) kurulum  2) veri indir (kadın konuşmacı seç)  3) veriyi hazırla  4) eğit  5) ONNX aktar + indir

In [ ]:
# 0) GPU kontrol
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "GPU yok"

In [ ]:
!pip install -q piper-training datasets
print("kuruldu")

In [ ]:
# 2) Common Voice ckb indir (kadın konuşmacı)
import os, json, librosa, soundfile as sf
from datasets import load_dataset
import numpy as np

os.makedirs("data", exist_ok=True)

try:
    ds = load_dataset("mozilla-foundation/common_voice_17_0", "ckb", split="train", streaming=False)
except Exception as e:
    print("17_0 denendi:", e)
    ds = load_dataset("mozilla-foundation/common_voice_13_0", "ckb", split="train")

df = ds.to_pandas()
# kadın konuşmacıları filtrele
f = df[df["gender"] == "female"]
print("kadın kayıt sayısı:", len(f))

# en çok kayıt veren tek kadın konuşmacıyı seç (tutarlı ses)
top = f["client_id"].value_counts().idxmax()
sub = f[f["client_id"] == top]
print("seçilen konuşmacı:", top, "kayıt:", len(sub))

# hedef süreyi ayarla (yaklaşık 3 sn / kayıt)
target = int(1.0 * 3600 / 3)  # 1 saat
sub = sub.head(target)

meta = []
for i, row in enumerate(sub.itertuples()):
    arr = row.audio["array"]
    sr = row.audio["sampling_rate"]
    a = librosa.resample(arr.astype("float32"), orig_sr=sr, target_sr=22050)
    if a.ndim > 1:
        a = a.mean(axis=1)
    p = f"data/cv_{i:05d}.wav"
    sf.write(p, a, 22050)
    open(p.replace(".wav", ".txt"), "w", encoding="utf-8").write(row.sentence)
    meta.append([p, row.sentence])

print("hazırlanan örnek:", len(meta))

In [ ]:
# 3) kontrol
import glob
print("wav:", len(glob.glob("data/*.wav")), "txt:", len(glob.glob("data/*.txt")))

In [ ]:
!piper.train_prepare --dataset-dir data --sample-rate 22050 --language ckb --config-out config.json
print("config ok")

In [ ]:
# 4) eğit (Colab GPU; ~1-3 saat veri boyutuna göre)
!piper.train --config-dir . --run-name sorani-female
print("eğitim bitti")

In [ ]:
# 5) ONNX aktar
!piper.export_onnx --run-dir runs --output-dir output --dataset-dir data
!ls -la output/

In [ ]:
# 6) indir
from google.colab import files
import glob
for f in sorted(glob.glob("output/*")):
    print("indir:", f)
    files.download(f)